# Purpose:
- Generate czstack id - hcr id matching table from the last landmarks file

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import json

from manual_coreg_utils import get_ids_from_landmarks

DATA_DIR = Path('/root/capsule/data/')
%load_ext autoreload
%autoreload 2

In [2]:
def get_czstack_id(x):
    if x.startswith('cz'):
        return int(x.split('-')[0][2:])
    else:
        return -1


def get_hcr_id(x):
    if x.startswith('cz'):
        return int(x.split('-')[1][3:])
    else:
        return -1

In [3]:
subject_id = 790322
czstack_date = '2025-07-10'
save_dir = Path(f'/root/capsule/scratch/{subject_id}_{czstack_date}_coreg_cpsam/')

# Auto pipeline: use final auto landmarks if available, else fall back to manual qced
auto_lm_path = save_dir / f'{subject_id}_{czstack_date}_landmarks_auto_final.csv'

# Manual pipeline: look for the last qced iteration file
import glob as _glob
qced_files = sorted(_glob.glob(str(save_dir / f'{subject_id}_{czstack_date}_landmarks_matched_ext_iter*_qced.csv')))
if not qced_files:
    # Also search in the coreg dir
    coreg_dir = Path(f'/root/capsule/data') / [
        d for d in Path('/root/capsule/data').iterdir()
        if f'{subject_id}' in d.name and 'ctl-czstack-hcr-coreg' in d.name
    ][0].name if any(
        f'{subject_id}' in d.name and 'ctl-czstack-hcr-coreg' in d.name
        for d in Path('/root/capsule/data').iterdir()
    ) else save_dir
    qced_files = sorted(_glob.glob(str(coreg_dir / f'*landmarks*qced*.csv')))

if auto_lm_path.exists():
    final_landmarks_file = auto_lm_path
    print(f'Using auto pipeline landmarks: {final_landmarks_file.name}')
elif qced_files:
    final_landmarks_file = Path(qced_files[-1])
    print(f'Using manual pipeline landmarks: {final_landmarks_file.name}')
else:
    raise FileNotFoundError(f'No final landmarks found in {save_dir}')

assert final_landmarks_file.exists(), f'{final_landmarks_file} does not exist'

In [4]:
save_fn = save_dir / f'{subject_id}_coreg_table.csv'

final_qced_landmarks = pd.read_csv(final_landmarks_file, header=None)
columns = ['ids', 'active', 'czstack_x', 'czstack_y', 'czstack_z', 'hcr_x', 'hcr_y', 'hcr_z']
assert len(final_qced_landmarks.columns) == len(columns), \
    f'Expected {len(columns)} columns, got {len(final_qced_landmarks.columns)}'
final_qced_landmarks.columns = columns

# Normalise 'active' column (may be string 'True'/'False' or bool)
if final_qced_landmarks['active'].dtype == object:
    final_qced_landmarks['active'] = final_qced_landmarks['active'].str.strip().str.lower().map(
        {'true': True, 'false': False}
    ).fillna(False)

# Handle both naming conventions:
#   cz{N}-hcr{M}       (standard)
#   qced_cz{N}-hcr{M}  (manual QC pipeline adds 'qced_' prefix)
prev_qced_mask = final_qced_landmarks.ids.str.startswith('qced')
if prev_qced_mask.any():
    prev_qced_ids = [qcid.replace('qced_', '') for qcid in final_qced_landmarks.loc[prev_qced_mask, 'ids'].values]
    final_qced_landmarks.loc[prev_qced_mask, 'active'] = True
    final_qced_landmarks.loc[prev_qced_mask, 'ids'] = prev_qced_ids

num_roi_points = (final_qced_landmarks.ids.str.startswith('cz')).sum()
final_active_landmarks = final_qced_landmarks.query('active').copy()

matched_landmarks = final_active_landmarks.copy()
matched_landmarks['czstack_id'] = matched_landmarks.ids.apply(get_czstack_id)
matched_landmarks['hcr_id'] = matched_landmarks.ids.apply(get_hcr_id)
matched_landmarks = matched_landmarks[matched_landmarks.czstack_id != -1]
assert matched_landmarks.active.sum() == len(matched_landmarks)
assert len(set(matched_landmarks.czstack_id)) == len(matched_landmarks), \
    'Duplicate czstack ids in matched landmarks!'
assert len(set(matched_landmarks.hcr_id)) == len(matched_landmarks), \
    'Duplicate hcr ids in matched landmarks!'
matched_landmarks = matched_landmarks.sort_values(by='czstack_id')[['czstack_id', 'hcr_id']].reset_index(drop=True)

# Merge per-match metadata if available (from auto pipeline)
match_meta_path = save_dir / f'{subject_id}_{czstack_date}_auto_match_metadata.csv'
if match_meta_path.exists():
    match_meta = pd.read_csv(match_meta_path)
    match_meta = match_meta.rename(columns={'czstack_cell_id': 'czstack_id', 'hcr_cell_id': 'hcr_id'})
    matched_landmarks = matched_landmarks.merge(
        match_meta[['czstack_id', 'hcr_id', 'iter_matched', 'match_probability']],
        on=['czstack_id', 'hcr_id'], how='left'
    )
    print(f'Merged per-match metadata (iter_matched, match_probability)')

# Final assurance with alternative method
czstack_ids, hcr_ids = get_ids_from_landmarks(final_active_landmarks)
check_df = pd.DataFrame({'czstack_id': czstack_ids, 'hcr_id': hcr_ids})
check_df = check_df.sort_values(by='czstack_id').reset_index(drop=True)
check_core = check_df[['czstack_id', 'hcr_id']].reset_index(drop=True)
matched_core = matched_landmarks[['czstack_id', 'hcr_id']].reset_index(drop=True)
assert check_core.equals(matched_core), 'Mismatch between two extraction methods!'

# Save
matched_landmarks.to_csv(save_fn, index=False)

# Report
print(f'Total number of czstack ROIs: {num_roi_points}')
print(f'Number of matched landmarks:  {len(matched_landmarks)}')
print(f'Matching rate:                {len(matched_landmarks)/num_roi_points:.2%}')
print(f'Saved to: {save_fn}')

# Create New Data